# Supplementary Figure S2: genomic coordinate balancing

**Purpose.** Recompute variant-type AUROCs after balancing the pathogenic and benign classes within 500 kbp genomic bins.

## Reproducibility contract

- **Run from:** the repository root.
- **Input:** `data/processed/clinvar_benchmark_updated.csv` (the full scored benchmark; not stored in GitHub).
- **Outputs:** `results/tables/figure_s2_coordinate_balance_auroc.csv` and `results/figures/figure_s2.svg`.
- **Randomness:** deterministic downsampling and 1,000 bootstrap replicates, seed 42.
- **Status:** reconstructed from the revised Methods; author confirmation is required before publication.


## 1. Setup


In [1]:
from pathlib import Path
import pandas as pd

# Run this notebook from the repository root.
REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data"
MODEL_SCORE_DIR = DATA_DIR / "model_scores"
RESULTS_DIR = REPO_ROOT / "results"
TABLE_DIR = RESULTS_DIR / "tables"
FIGURE_DIR = RESULTS_DIR / "figures"
SCORED_BENCHMARK_PATH = DATA_DIR / "processed" / "clinvar_benchmark_updated.csv"
RANDOM_SEED = 42

def read_table(path, required_columns=()):
    """Read a local table and verify that required columns are present."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}. See README.md.")
    frame = pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(
        path, low_memory=False, dtype={"#CHROM": str}
    )
    missing = sorted(set(required_columns) - set(frame.columns))
    if missing:
        raise ValueError(f"{path} is missing required columns: {missing}")
    return frame

def validate_output_files(paths):
    """Verify that every expected output was created and is not empty."""
    for value in paths:
        path = Path(value)
        if not path.exists() or path.stat().st_size == 0:
            raise RuntimeError(f"Expected output was not created: {path}")
    print(f"Validated {len(paths)} output file(s).")

HIGHER_IS_MORE_PATHOGENIC = {
    "PhyloP", "DNABERT2", "AlphaMissense", "PrimateAI_3D",
    "AlphaGenome_quantile", "Rule_based", "ntv3_post_log2fc_max",
}

DNA_MODEL_COLUMNS = [
    "PhyloP", "Evo2_7B", "Evo2_40B", "AlphaGenome_quantile", "Rule_based",
    "GPN_MSA", "PhyloGPN", "gpnstar_v_llr", "DNABERT2",
    "ntv3_pre_position_llr", "ntv3_post_log2fc_max",
]
ESM_MODEL_COLUMNS = ["ESM1b", "ESM1v", "ESM2", "vesm_score"]
MODEL_COLUMNS = DNA_MODEL_COLUMNS + ESM_MODEL_COLUMNS + ["AlphaMissense", "PrimateAI_3D"]

SUBGROUP_COLUMNS = [
    "group: missense", "group: missense + 3'UTR",
    "group: missense + intron (non-splice)", "group: stop gain",
    "group: start loss", "group: noncoding", "group: stop loss",
    "group: synonymous", "group: 5'UTR", "group: 3'UTR",
    "group: 3'UTR + RNA gene", "group: splice",
    "group: intron (non-splice)", "group: RNA gene",
]
SUBGROUP_DISPLAY_NAMES = {
    "group: missense": "Missense", "group: missense + 3'UTR": "Missense & 3' UTR",
    "group: missense + intron (non-splice)": "Missense & Intron (Non-splice)",
    "group: stop gain": "Stop gain", "group: start loss": "Start loss",
    "group: noncoding": "Noncoding", "group: stop loss": "Stop loss",
    "group: synonymous": "Synonymous", "group: 5'UTR": "5' UTR",
    "group: 3'UTR": "3' UTR", "group: 3'UTR + RNA gene": "3' UTR + RNA gene",
    "group: splice": "Splice", "group: intron (non-splice)": "Intron (Non-splice)",
    "group: RNA gene": "RNA gene",
}
MODEL_DISPLAY_NAMES = {
    "PhyloP": "PhyloP", "Evo2_7B": "Evo2 7B", "Evo2_40B": "Evo2 40B",
    "AlphaGenome_quantile": "AlphaGenome", "Rule_based": "Rule-based",
    "GPN_MSA": "GPN-MSA", "PhyloGPN": "PhyloGPN", "gpnstar_v_llr": "GPN-Star",
    "DNABERT2": "DNABERT2", "ntv3_pre_position_llr": "NTv3-pre",
    "ntv3_post_log2fc_max": "NTv3-post", "ESM1b": "ESM1b", "ESM1v": "ESM1v",
    "ESM2": "ESM2", "vesm_score": "VESM++", "AlphaMissense": "AlphaMissense",
    "PrimateAI_3D": "PrimateAI-3D",
}

def models_for_subgroup(subgroup):
    if subgroup in {
        "group: missense", "group: missense + 3'UTR",
        "group: missense + intron (non-splice)", "group: stop gain",
    }:
        return MODEL_COLUMNS.copy()
    if subgroup == "group: start loss":
        return DNA_MODEL_COLUMNS + ESM_MODEL_COLUMNS
    return DNA_MODEL_COLUMNS.copy()

def coordinate_balanced_sample(frame, bin_size=500_000, seed=RANDOM_SEED):
    """Downsample the majority class within each genomic coordinate bin."""
    work = frame.copy()
    work["_coordinate_bin"] = (
        work["#CHROM"].astype(str).str.replace("chr", "", regex=False)
        + ":" + (pd.to_numeric(work["POS"]) // bin_size).astype(str)
    )
    rng = np.random.default_rng(seed)
    kept = []
    for _, bin_frame in work.groupby("_coordinate_bin", sort=True):
        classes = {
            int(label): group.index.to_numpy()
            for label, group in bin_frame.groupby("ClinVar_label")
        }
        if set(classes) != {0, 1}:
            continue
        size = min(len(classes[0]), len(classes[1]))
        kept.extend([
            rng.choice(classes[0], size=size, replace=False),
            rng.choice(classes[1], size=size, replace=False),
        ])
    if not kept:
        return work.iloc[0:0].drop(columns="_coordinate_bin")
    return work.loc[np.concatenate(kept)].drop(columns="_coordinate_bin").sort_index()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from sklearn.metrics import roc_auc_score


FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)


## 2. Load and balance input


In [2]:
data = read_table(SCORED_BENCHMARK_PATH, ["#CHROM", "POS", "ClinVar_label"] + SUBGROUP_COLUMNS)
balanced = coordinate_balanced_sample(data, bin_size=500_000, seed=RANDOM_SEED)
print(f"Balanced sample: {len(balanced):,} of {len(data):,} variants")


Balanced sample: 111,508 of 242,132 variants


## 3. Compute AUROC and confidence intervals


In [3]:
N_BOOTSTRAPS = 1_000


def bootstrap_auroc(labels, scores, n_boot=N_BOOTSTRAPS, seed=RANDOM_SEED):
    labels = np.asarray(labels, dtype=int)
    scores = np.asarray(scores, dtype=float)
    estimate = roc_auc_score(labels, scores)
    rng = np.random.default_rng(seed)
    class_indices = [np.flatnonzero(labels == label) for label in (0, 1)]
    draws = []
    for _ in range(n_boot):
        index = np.concatenate([
            rng.choice(indices, size=len(indices), replace=True)
            for indices in class_indices
        ])
        draws.append(roc_auc_score(labels[index], scores[index]))
    low, high = np.percentile(draws, [2.5, 97.5])
    return float(estimate), float(low), float(high)


rows = []
for subgroup in SUBGROUP_COLUMNS:
    subgroup_frame = balanced[balanced[subgroup] == 1]
    for model in models_for_subgroup(subgroup):
        if model not in subgroup_frame.columns:
            continue
        work = subgroup_frame[["ClinVar_label", model]].dropna()
        if work["ClinVar_label"].nunique() != 2:
            continue
        score = work[model] if model in HIGHER_IS_MORE_PATHOGENIC else -work[model]
        estimate, low, high = bootstrap_auroc(work["ClinVar_label"], score)
        counts = work["ClinVar_label"].value_counts()
        rows.append({
            "subgroup": subgroup,
            "model": model,
            "n": len(work),
            "n_benign": int(counts.get(0, 0)),
            "n_pathogenic": int(counts.get(1, 0)),
            "auroc": estimate,
            "ci_low": low,
            "ci_high": high,
        })

results = pd.DataFrame(rows)
results.to_csv(TABLE_DIR / "figure_s2_coordinate_balance_auroc.csv", index=False)
results.head()


,subgroup,model,n,n_benign,n_pathogenic,auroc,ci_low,ci_high
0,group: missense,PhyloP,19583,8096,11487,0.854921,0.849055,0.860128
1,group: missense,Evo2_7B,19583,8096,11487,0.815946,0.809975,0.822271
2,group: missense,Evo2_40B,19583,8096,11487,0.827325,0.821458,0.833285
3,group: missense,AlphaGenome_quantile,19583,8096,11487,0.562408,0.554018,0.570417
4,group: missense,Rule_based,19583,8096,11487,0.500000,0.500000,0.500000


## 4. Render the manuscript figure


In [4]:
palette = {
    "PhyloP": "#808080", "Evo2_7B": "#e56b6f", "Evo2_40B": "#c93735",
    "AlphaGenome_quantile": "#a2d94d", "Rule_based": "#b0b0b0",
    "GPN_MSA": "#859ed7", "PhyloGPN": "#f47f1e", "gpnstar_v_llr": "#20b2aa",
    "DNABERT2": "#1b78b2", "ntv3_pre_position_llr": "#6baed6",
    "ntv3_post_log2fc_max": "#54278f", "ESM1b": "#2da248", "ESM1v": "#8f69c5",
    "ESM2": "#22bdd2", "vesm_score": "#e377c2", "AlphaMissense": "#333a8c",
    "PrimateAI_3D": "#d26101",
}

fig, axes = plt.subplots(5, 3, figsize=(12, 15), dpi=300)
axes = axes.T.flatten()
for axis, subgroup in zip(axes, SUBGROUP_COLUMNS):
    panel = results[results["subgroup"] == subgroup].sort_values("auroc")
    unreliable = panel.empty or min(panel["n_benign"].min(), panel["n_pathogenic"].min()) < 10
    alpha = 0.3 if unreliable else 1.0
    for y, row in enumerate(panel.itertuples()):
        axis.barh(y, row.auroc, color=palette.get(row.model, "#777777"), alpha=alpha)
        xerr_low = max(row.auroc - row.ci_low, 0.0)
        xerr_high = max(row.ci_high - row.auroc, 0.0)
        axis.errorbar(row.auroc, y, xerr=[[xerr_low], [xerr_high]], fmt="none", color="black", lw=0.7, capsize=2, alpha=alpha)
    axis.set_yticks(range(len(panel)), [MODEL_DISPLAY_NAMES.get(model, model) for model in panel["model"]], fontsize=7)
    axis.set_title(SUBGROUP_DISPLAY_NAMES[subgroup], fontsize=8, fontweight="bold")
    axis.set_xlim(0.5, 1.02)
    axis.xaxis.set_major_locator(ticker.MultipleLocator(0.1))
    axis.tick_params(axis="x", labelsize=7)
    axis.spines[["top", "right"]].set_visible(False)
for axis in axes[len(SUBGROUP_COLUMNS):]:
    axis.axis("off")
fig.suptitle("Coordinate-balanced AUROC by variant-type subgroup", fontsize=10, fontweight="bold")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "figure_s2.svg", format="svg", bbox_inches="tight")
plt.show()


## 5. Validate outputs


In [5]:

validate_output_files([
    TABLE_DIR / "figure_s2_coordinate_balance_auroc.csv",
    FIGURE_DIR / "figure_s2.svg",
])


Validated 2 output file(s).
